In [1]:
import pandas as pd

file_path = "data/yellow_tripdata_2021-01.csv.gz"

df = pd.read_csv(file_path, nrows=5)

print(df)
print(df.columns.tolist())

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [2]:
import pandas as pd

file_path = "data/yellow_tripdata_2021-01.csv.gz"

chunksize = 100_000
total_rows = 0
profile_data = {}

for chunk in pd.read_csv(file_path, chunksize=chunksize):
    total_rows += len(chunk)

    for col in chunk.columns:
        if col not in profile_data:
            profile_data[col] = {
                "dtype": str(chunk[col].dtype),
                "null_count": 0,
                "min": None,
                "max": None
            }

        profile_data[col]["null_count"] += int(
            chunk[col].isna().sum()
        )

        col_min = chunk[col].min()
        col_max = chunk[col].max()

        if pd.notna(col_min):
            old_min = profile_data[col]["min"]
            if old_min is None or col_min < old_min:
                profile_data[col]["min"] = col_min

        if pd.notna(col_max):
            old_max = profile_data[col]["max"]
            if old_max is None or col_max > old_max:
                profile_data[col]["max"] = col_max

profile = pd.DataFrame.from_dict(profile_data, orient="index")
profile["null_percent"] = (
    profile["null_count"] / total_rows * 100
)

print("Total rows:", total_rows)
display(profile)

C:\Users\User\AppData\Local\Temp\ipykernel_18004\408366413.py:9: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunksize):


Total rows: 1369765


,dtype,null_count,min,max,null_percent
VendorID,int64,98352,1,2,7.18021
tpep_pickup_datetime,str,0,2008-12-31 23:05:14,2021-02-22 16:52:16,0.00000
tpep_dropoff_datetime,str,0,2008-12-31 23:07:22,2021-02-22 16:56:15,0.00000
passenger_count,int64,98352,0,8,7.18021
trip_distance,float64,0,0.0,263163.28,0.00000
RatecodeID,int64,98352,1,99,7.18021
store_and_fwd_flag,str,98352,N,Y,7.18021
PULocationID,int64,0,1,265,0.00000
DOLocationID,int64,0,1,265,0.00000
payment_type,int64,98352,1,4,7.18021


In [3]:
import pandas as pd

file_path = "data/yellow_tripdata_2021-01.csv.gz"

counts = {
    "pickup_missing_or_invalid": 0,
    "pickup_outside_january": 0,
    "dropoff_missing_or_invalid": 0,
    "dropoff_outside_january": 0,
    "fare_zero_or_negative": 0,
    "distance_zero_or_negative": 0,
    "passenger_count_zero": 0,
    "exact_duplicates_within_chunks": 0,
}

for chunk in pd.read_csv(file_path, chunksize=100_000):

    # Convert dates to datetime
    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"], errors="coerce"
    )
    dropoff = pd.to_datetime(
        chunk["tpep_dropoff_datetime"], errors="coerce"
    )

    # Check pickup dates
    counts["pickup_missing_or_invalid"] += int(pickup.isna().sum())
    counts["pickup_outside_january"] += int(
        (pickup.notna() & ~pickup.between(
            "2021-01-01", "2021-01-31 23:59:59.999999"
        )).sum()
    )

    # Check drop-off dates
    counts["dropoff_missing_or_invalid"] += int(dropoff.isna().sum())
    counts["dropoff_outside_january"] += int(
        (dropoff.notna() & ~dropoff.between(
            "2021-01-01", "2021-01-31 23:59:59.999999"
        )).sum()
    )

    # Check unusual values
    counts["fare_zero_or_negative"] += int(
        (chunk["fare_amount"] <= 0).sum()
    )
    counts["distance_zero_or_negative"] += int(
        (chunk["trip_distance"] <= 0).sum()
    )
    counts["passenger_count_zero"] += int(
        (chunk["passenger_count"] == 0).sum()
    )

    # Exact duplicates within this chunk
    counts["exact_duplicates_within_chunks"] += int(
        chunk.duplicated().sum()
    )

print(pd.Series(counts, name="problem_count"))

C:\Users\User\AppData\Local\Temp\ipykernel_18004\3553334128.py:16: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=100_000):


pickup_missing_or_invalid             0
pickup_outside_january               24
dropoff_missing_or_invalid            0
dropoff_outside_january             137
fare_zero_or_negative              7411
distance_zero_or_negative         19952
passenger_count_zero              26726
exact_duplicates_within_chunks        0
Name: problem_count, dtype: int64


In [4]:
for chunk in pd.read_csv(
    file_path,
    chunksize=100_000,
    dtype={"store_and_fwd_flag": "string"}
):
    # Your analysis code goes here
    pass

In [5]:
def clean(df):
    cleaned = df.copy()
    removed = {}

    # 1. Remove missing or invalid pickup dates
    pickup = pd.to_datetime(
        cleaned["tpep_pickup_datetime"],
        errors="coerce"
    )

    valid_pickup = pickup.between(
        "2021-01-01",
        "2021-01-31 23:59:59.999999"
    )

    before = len(cleaned)
    cleaned = cleaned.loc[valid_pickup].copy()
    removed["invalid_or_outside_pickup"] = before - len(cleaned)

    # 2. Remove exact duplicate rows
    before = len(cleaned)
    cleaned = cleaned.drop_duplicates().copy()
    removed["duplicate_rows"] = before - len(cleaned)

    # 3. Flag suspicious values without deleting them
    cleaned["flag_nonpositive_fare"] = (
        cleaned["fare_amount"] <= 0
    )

    cleaned["flag_nonpositive_distance"] = (
        cleaned["trip_distance"] <= 0
    )

    cleaned["flag_zero_passengers"] = (
        cleaned["passenger_count"] == 0
    )

    # 4. Flag drop-off dates outside January
    dropoff = pd.to_datetime(
        cleaned["tpep_dropoff_datetime"],
        errors="coerce"
    )

    cleaned["flag_dropoff_outside_january"] = (
        ~dropoff.between(
            "2021-01-01",
            "2021-01-31 23:59:59.999999"
        )
    )

    removed["total_rows_removed"] = len(df) - len(cleaned)

    print("Rows before:", len(df))
    print("Rows after:", len(cleaned))
    print("Removal counts:", removed)
    print("Flagged nonpositive fares:",
          int(cleaned["flag_nonpositive_fare"].sum()))
    print("Flagged nonpositive distances:",
          int(cleaned["flag_nonpositive_distance"].sum()))
    print("Flagged zero passengers:",
          int(cleaned["flag_zero_passengers"].sum()))

    return cleaned, removed


In [6]:
sample = pd.read_csv(
    file_path,
    nrows=1000,
    dtype={"store_and_fwd_flag": "string"}
)

cleaned_sample, removed = clean(sample)

cleaned_sample.head()

Rows before: 1000
Rows after: 994
Removal counts: {'invalid_or_outside_pickup': 6, 'duplicate_rows': 0, 'total_rows_removed': 6}
Flagged nonpositive fares: 10
Flagged nonpositive distances: 19
Flagged zero passengers: 16


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,flag_nonpositive_fare,flag_nonpositive_distance,flag_zero_passengers,flag_dropoff_outside_january
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1,2.10,1,N,142,43,2,...,0.5,0.00,0.0,0.3,11.80,2.5,False,False,False,False
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1,0.20,1,N,238,151,2,...,0.5,0.00,0.0,0.3,4.30,0.0,False,False,False,False
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1,14.70,1,N,132,165,1,...,0.5,8.65,0.0,0.3,51.95,0.0,False,False,False,False
3,1,2021-01-01 00:15:48,2021-01-01 00:31:01,0,10.60,1,N,138,132,1,...,0.5,6.05,0.0,0.3,36.35,0.0,False,False,True,False
4,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1,4.94,1,N,68,33,1,...,0.5,4.06,0.0,0.3,24.36,2.5,False,False,False,False


In [7]:
from pathlib import Path
import pandas as pd

output_path = Path("cleaned_taxi_data.csv")
if output_path.exists():
    output_path.unlink()

total_before = 0
total_after = 0
total_removed = {}

for chunk in pd.read_csv(
    file_path,
    chunksize=100_000,
    dtype={"store_and_fwd_flag": "string"}
):
    total_before += len(chunk)

    cleaned_chunk, removed = clean(chunk)
    total_after += len(cleaned_chunk)

    for rule, count in removed.items():
        total_removed[rule] = (
            total_removed.get(rule, 0) + count
        )

    cleaned_chunk.to_csv(
        output_path,
        mode="a",
        index=False,
        header=not output_path.exists() or output_path.stat().st_size == 0
    )

print("Total rows before:", total_before)
print("Total rows after:", total_after)
print("Total rows removed:", total_before - total_after)
print("Removal counts:", total_removed)

Rows before: 100000
Rows after: 99984
Removal counts: {'invalid_or_outside_pickup': 16, 'duplicate_rows': 0, 'total_rows_removed': 16}
Flagged nonpositive fares: 721
Flagged nonpositive distances: 1461
Flagged zero passengers: 1973
Rows before: 100000
Rows after: 100000
Removal counts: {'invalid_or_outside_pickup': 0, 'duplicate_rows': 0, 'total_rows_removed': 0}
Flagged nonpositive fares: 582
Flagged nonpositive distances: 1784
Flagged zero passengers: 2027
Rows before: 100000
Rows after: 100000
Removal counts: {'invalid_or_outside_pickup': 0, 'duplicate_rows': 0, 'total_rows_removed': 0}
Flagged nonpositive fares: 563
Flagged nonpositive distances: 1570
Flagged zero passengers: 2319
Rows before: 100000
Rows after: 99996
Removal counts: {'invalid_or_outside_pickup': 4, 'duplicate_rows': 0, 'total_rows_removed': 4}
Flagged nonpositive fares: 563
Flagged nonpositive distances: 1648
Flagged zero passengers: 2221
Rows before: 100000
Rows after: 100000
Removal counts: {'invalid_or_outside_